# Exercise 2.2.5: Merging & Combining Datasets
*Exercício 2.2.5: Juntar e Combinar Conjuntos de Dados*

This exercise reads the feature table from **Exercise 2.2.4** (`data/20_processed/datania_households_features.csv`) and enriches it by joining reference tables, then stacking survey waves.

*Este exercício lê a tabela de variáveis do **Exercício 2.2.4** (`data/20_processed/datania_households_features.csv`) e enriquece-a juntando tabelas de referência e, depois, empilhando vagas do inquérito.*

You will practice:
- Combining tables with `pd.merge()` and choosing the right **join type**
- **Join-key hygiene**: matching dtypes, trimming whitespace, counting missing keys
- Auditing a merge with `indicator=True` and enforcing expectations with `validate=`
- Recognising **cardinality** problems (row explosions, many-to-many merges)
- **Post-merge validation** and saving the final table
- Stacking waves/modules with `pd.concat()`

*Vai praticar:*
- *Combinar tabelas com `pd.merge()` e escolher o **tipo de junção** correto*
- ***Higiene da chave de junção**: tipos compatíveis, remoção de espaços, contagem de chaves em falta*
- *Auditar uma junção com `indicator=True` e impor expectativas com `validate=`*
- *Reconhecer problemas de **cardinalidade** (explosão de linhas, junções muitos-para-muitos)*
- ***Validação pós-junção** e gravação da tabela final*
- *Empilhar vagas ou módulos com `pd.concat()`*

> **Pipeline:** run Exercises 2.2.3 and 2.2.4 first. This notebook writes the final merged table to `20_processed/`.

> ***Fluxo de trabalho:** execute primeiro os Exercícios 2.2.3 e 2.2.4. Este notebook escreve a tabela final juntada em `20_processed/`.*

### Path Setup (run first)
*Configuração do caminho (execute primeiro)*

In [ ]:
import os
import numpy as np
import pandas as pd

# Load the feature table from 2.2.4 | Carregar a tabela de variáveis do 2.2.4
features_path = '../../data/20_processed/datania_households_features.csv'
df = pd.read_csv(features_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Households:', df.shape)
df[['hh_id', 'region_code', 'province_name', 'district', 'education_code']].head()

---

## Task 1: A basic left merge with a region lookup
*Tarefa 1: Uma junção à esquerda simples com uma tabela de regiões*

Geographic labels usually live in a separate reference table. Build a small `region_lookup` and attach `region_name` to every household with a **left join** (keep all households).

*Os rótulos geográficos costumam estar numa tabela de referência separada. Construa uma pequena `region_lookup` e associe `region_name` a cada agregado familiar com uma **junção à esquerda** (mantendo todos os agregados).*

In [ ]:
# Reference table with one row per region | Tabela de referência com uma linha por região
region_lookup = pd.DataFrame({
    'region_code': ['01', '02', '03', '04', '05', '06'],
    'region_name': ['Eastern', 'Northern', 'Central', 'Southern', 'Western', 'Highland'],
})

merged = pd.merge(df, region_lookup, on='region_code', how= # your code here: 'left' | o seu código aqui: 'left' )
print('Before:', df.shape, '-> After:', merged.shape)
merged[['hh_id', 'region_code', 'region_name']].head(10)

**Questions:**

- Did the row count change? For a left join with a unique key on the right, what should happen to it?
- Some households get a `NaN` `region_name`. Which households, and why (think about what 2.2.3 did to `region_code`)?

***Perguntas:***

- *O número de linhas mudou? Numa junção à esquerda com uma chave única do lado direito, o que deveria acontecer-lhe?*
- *Alguns agregados ficam com `region_name` a `NaN`. Quais e porquê (pense no que o 2.2.3 fez a `region_code`)?*

---

## Task 2: Join-key hygiene
*Tarefa 2: Higiene da chave de junção*

Most "failed merge" mysteries are a key that is text on one side and a number on the other, or a stray space. Before joining, make both keys the **same dtype**, trim whitespace, and count missing keys on each side.

*A maioria dos mistérios de "junção falhada" resume-se a uma chave que é texto de um lado e número do outro, ou a um espaço perdido. Antes de juntar, coloque as duas chaves com o **mesmo dtype**, retire os espaços e conte as chaves em falta de cada lado.*

In [ ]:
# Same dtype and no stray spaces on both sides
# Mesmo dtype e sem espaços perdidos nos dois lados
df['region_code'] = df['region_code'].astype('string').str.strip()
region_lookup['region_code'] = region_lookup['region_code'].astype('string').str.strip()

print('Missing keys (households):', df['region_code']. # your code here: isna().sum() | o seu código aqui )
print('Missing keys (lookup):    ', region_lookup['region_code'].isna().sum())

**Questions:**

- How many household rows have a missing join key? After a left join, what value will their `region_name` take?
- Why is fixing the key *at the source* better than dropping the unmatched rows afterwards?

***Perguntas:***

- *Quantas linhas de agregados têm a chave de junção em falta? Depois de uma junção à esquerda, que valor terá o seu `region_name`?*
- *Porque é que corrigir a chave *na origem* é melhor do que eliminar depois as linhas sem correspondência?*

---

## Task 3: Choose the join type, then audit with `indicator`
*Tarefa 3: Escolher o tipo de junção e auditar com `indicator`*

The join type decides which rows survive. Add `indicator=True` to label every row as `left_only`, `right_only`, or `both`.

*O tipo de junção decide que linhas sobrevivem. Acrescente `indicator=True` para rotular cada linha como `left_only`, `right_only` ou `both`.*

In [ ]:
# Label where each row came from | Rotular a origem de cada linha
merged = pd.merge(df, region_lookup, on='region_code', how='left', indicator= # your code here | o seu código aqui )
print(merged['_merge'].value_counts())

In [ ]:
# Compare the join types side by side | Comparar os tipos de junção lado a lado
inner = pd.merge(df, region_lookup, on='region_code', how='inner')
outer = pd.merge(df, region_lookup, on='region_code', how='outer')
print('inner:', inner.shape[0], '| left:', merged.shape[0], '| outer:', outer.shape[0])

**Questions:**

- What does each `left_only` row represent here? Why is that group worth reporting back to the data team?
- The inner join has fewer rows than the left join. Which households did it drop?

***Perguntas:***

- *O que representa aqui cada linha `left_only`? Porque vale a pena reportar esse grupo à equipa de dados?*
- *A junção interna tem menos linhas do que a junção à esquerda. Que agregados foram eliminados?*

---

## Task 4: Cardinality: row explosions and `validate`
*Tarefa 4: Cardinalidade: explosão de linhas e `validate`*

If the right-hand key is **not unique**, every duplicate match multiplies rows. `validate=` makes your assumption explicit and raises an error if it is violated.

*Se a chave do lado direito **não for única**, cada correspondência duplicada multiplica as linhas. `validate=` torna o seu pressuposto explícito e levanta um erro se este for violado.*

In [ ]:
# A lookup that accidentally lists region '01' twice
# Uma tabela de referência que lista a região '01' duas vezes por engano
bad_lookup = pd.DataFrame({
    'region_code': ['01', '01', '02'],
    'region_name': ['Eastern', 'Eastern (dup)', 'Northern'],
})

exploded = pd.merge(df, bad_lookup, on='region_code', how='left')
print('Rows before:', len(df), '-> after bad merge:', len(exploded))

In [ ]:
# Make the expectation explicit: each household should match at most one region.
# Torne a expectativa explícita: cada agregado deve corresponder no máximo a uma região.
try:
    pd.merge(df, bad_lookup, on='region_code', how='left', validate= # your code here: 'many_to_one' | o seu código aqui )
except Exception as e:
    print(type(e).__name__, '->', e)

**Questions:**

- Why did the row count grow? Which households were duplicated?
- What do `'one_to_one'`, `'one_to_many'`, and `'many_to_one'` each promise? Which fits a household -> region lookup?

***Perguntas:***

- *Porque é que o número de linhas aumentou? Que agregados foram duplicados?*
- *O que garante cada um de `'one_to_one'`, `'one_to_many'` e `'many_to_one'`? Qual se aplica a uma consulta agregado -> região?*

---

## Task 5: Merge on differently-named keys
*Tarefa 5: Juntar com chaves de nomes diferentes*

When the key column has a different name on each side, use `left_on` / `right_on`. Attach a full education label from a lookup keyed on `code`.

*Quando a coluna-chave tem um nome diferente em cada lado, use `left_on` / `right_on`. Associe um rótulo completo de educação a partir de uma tabela cuja chave é `code`.*

In [ ]:
# The key is called education_code on the left and code on the right
# A chave chama-se education_code à esquerda e code à direita
education_lookup = pd.DataFrame({
    'code': [1, 2, 3, 4],
    'education_label_full': ['No schooling', 'Primary', 'Secondary', 'Tertiary'],
})

merged_edu = pd.merge(
    df,
    education_lookup,
    left_on= # your code here: 'education_code' | o seu código aqui: 'education_code'
    right_on='code',
    how='left',
)
merged_edu[['hh_id', 'education_code', 'education_label_full']].head()

**Question:** `education_code` is a float (it carries `NaN`) while `code` is an integer. The merge still matches: why? When would a dtype mismatch silently produce *no* matches instead?

***Pergunta:** `education_code` é um float (porque contém `NaN`) enquanto `code` é um inteiro. A junção continua a encontrar correspondências: porquê? Em que situação é que uma incompatibilidade de dtype produziria antes, silenciosamente, *nenhuma* correspondência?*

---

## Task 6: Many-to-many: a warning
*Tarefa 6: Muitos-para-muitos: um aviso*

When **both** sides have duplicate keys, pandas produces the full cross-product, often nonsense. The fix is a safer (compound) key, not "merge harder".

*Quando **ambos** os lados têm chaves duplicadas, o pandas produz o produto cartesiano completo, muitas vezes sem sentido. A solução é uma chave mais segura (composta), e não "juntar com mais força".*

In [ ]:
# Two rows on each side, duplicate keys on both | Duas linhas de cada lado, chaves duplicadas em ambos
roster = pd.DataFrame({'hh_id': [1, 1], 'person': ['A', 'B']})
visits = pd.DataFrame({'hh_id': [1, 1], 'visit': ['V1', 'V2']})

mm = pd.merge(roster, visits, on='hh_id')
print(len(mm), 'rows from two 2-row tables')
mm

**Questions:**

- Why did two people and two visits become four rows?
- What compound key would make each row identify exactly one record?

***Perguntas:***

- *Porque é que duas pessoas e duas visitas se transformaram em quatro linhas?*
- *Que chave composta faria com que cada linha identificasse exatamente um registo?*

---

## Task 7: Post-merge validation, then save the final table
*Tarefa 7: Validação pós-junção e gravação da tabela final*

A merge is finished only when you have confirmed the result. Check the row count, key uniqueness, and the unmatched rate, then save to `20_processed/`.

*Uma junção só está terminada quando o resultado foi confirmado. Verifique o número de linhas, a unicidade da chave e a taxa de linhas sem correspondência e depois guarde em `20_processed/`.*

In [ ]:
# Confirm the result before saving | Confirmar o resultado antes de guardar
final = pd.merge(df, region_lookup, on='region_code', how='left')

print('Before:', len(df), '| After:', len(final))
print('Duplicate hh_id:', final['hh_id'].duplicated().sum())
print('Unmatched region:', round(final['region_name'].isna().mean(), 3))

In [ ]:
final = final.reset_index(drop=True)
out_path = '../../data/20_processed/datania_households_merged.csv'

final.to_csv( # your code here: index=False | o seu código aqui: index=False )
print('Saved:', out_path, '|', final.shape)

---

## Task 8: Appending waves with `concat()`
*Tarefa 8: Acrescentar vagas com `concat()`*

Merging combines **columns**; appending combines **rows**. When you collect the same survey across waves, stack them with `pd.concat()` and add a column recording where each row came from.

*Juntar combina **colunas**; acrescentar combina **linhas**. Quando recolhe o mesmo inquérito em várias vagas, empilhe-as com `pd.concat()` e acrescente uma coluna a registar a origem de cada linha.*

In [ ]:
# Two waves of the same survey | Duas vagas do mesmo inquérito
wave1 = df[['hh_id', 'region_code', 'income_dkw']].head(3).assign(wave='2024')
wave2 = pd.DataFrame({
    'hh_id': ['HH0101', 'HH0102'],
    'region_code': ['01', '02'],
    'income_dkw': [62000.0, 47000.0],
}).assign(wave='2025')

combined = pd.concat([wave1, wave2], ignore_index= # your code here: True | o seu código aqui: True )
combined

**Questions:**

- Why add a `wave` column before stacking?
- `concat()` aligns on column names and fills gaps with `NaN` without warning. What would happen if `wave2` had `income` instead of `income_dkw`? How do you guard against that schema drift?

***Perguntas:***

- *Porquê acrescentar uma coluna `wave` antes de empilhar?*
- *`concat()` alinha pelos nomes das colunas e preenche as lacunas com `NaN` sem avisar. O que aconteceria se `wave2` tivesse `income` em vez de `income_dkw`? Como se protege contra essa divergência de esquema?*